# Reproducing CME FedWatch
**Erian Stanley Yogaraj · Pricing the Fed · 24 November 2026**

What probability did the 11 November settlements imply for a quarter-point increase in December 2026? I obtain 86.3% using the current EFFR as the reference. The calculation below checks the one-meeting formula and the two-meeting tree against CME’s published examples before applying both to the team’s prices.

## 1. November 2026 settlements
Read Bloomberg daily settlements from the team CSV. December and January contain decisions. February is the first full month without a scheduled decision. All rates are in percent, changes in basis points and probabilities in decimals.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

data_dir = Path(r'C:\Users\syeri\OneDrive\Bureau\ESILV\A5\PI2\datacollect3_1copie\datacollect3_1copie')
if not data_dir.is_dir():
    data_dir = Path.cwd()
csv = data_dir / 'fed_funds_futures_history.csv'
if not csv.exists():
    csv = data_dir / 'fed_funds_futures_history(1).csv'
history = pd.read_csv(csv, usecols=['contract', 'date', 'px_settle'], parse_dates=['date'])

asof = pd.Timestamp('2026-11-11')
display(history.loc[history.date.eq(asof) & history.contract.isin(['FFZ26','FFF27','FFG27'])].sort_values('contract'))

,date,px_settle,contract
243887,2026-11-11,95.815,FFF27
243889,2026-11-11,95.675,FFG27
243910,2026-11-11,95.915,FFZ26


### Separating the days before and after the decision
The monthly implied EFFR equals 100 minus the settlement. The decision day belongs to the starting rate, and the new rate applies from the next calendar day. The function solves the weighted average for the missing endpoint and brackets the expected change between adjacent multiples of 25 bp.

In [6]:
def meeting_rates(price, meeting, anchor, side):
    day = pd.Timestamp(meeting)
    n = day.day
    total = day.days_in_month
    m = total - n
    average = 100 - price
    if side == 'end':
        start, end = (total * average - m * anchor) / n, anchor
    elif side == 'start' and m > 0:
        start, end = anchor, (total * average - n * anchor) / m
    else:
        raise ValueError('Check the anchor side and the number of post-decision days.')
    return average, start, end

def move_weights(change_bp):
    # The floor also gives the lower signed outcome for cuts.
    units = round(change_bp / 25, 12)
    lower = int(np.floor(units))
    fraction = units - lower
    weights = {25 * lower: 1 - fraction}
    if fraction:
        weights[25 * (lower + 1)] = fraction
    return weights

## 2. Validate the one-meeting calculation
In CME’s September 2022 example, 21 days carry the starting rate and nine days carry the ending rate. October has no scheduled meeting, so its quote of 96.9400 implies a September ending EFFR of 3.0600%. The September quote of 97.4475 implies a 2.5525% monthly average. Solving the weighted average backwards gives the initial rate.

In [9]:
september, november = '2022-09-21', '2022-11-02'
avg, start, end = meeting_rates(97.4475, september, 100 - 96.9400, 'end')
weights = move_weights(100 * (end - start))
np.testing.assert_allclose([avg, start, end, weights[50], weights[75]], [2.5525, 2.335, 3.06, .10, .90])
print(f'September: {start:.4f}% to {end:.4f}% | expected move {100*(end-start):.1f} bp')
display(pd.Series(weights, name='Probability').rename_axis('September move (bp)').to_frame())

September: 2.3350% to 3.0600% | expected move 72.5 bp


,Probability
September move (bp),
50,0.1
75,0.9


### Reading the two outcomes
The expected increase of 72.5 bp falls nine-tenths of the way from +50 to +75 bp. Assigning 90% to +75 bp reproduces the price **under the two-adjacent-outcome assumption**. A single futures price gives a mean. Allowing three or more arbitrary outcomes would leave their individual probabilities undetermined.

## 3. Validate the two-meeting tree
CME’s hypothetical June–July example isolates the choice of probability view. The published implied levels are 5.125%, 5.250% and 5.375%. Under the conditional tree, either meeting can deliver a hike; two paths lead to a cumulative +25 bp. The Aggregated calculation brackets July’s total expected change from May directly.

In [13]:
from collections import defaultdict

def extend_tree(prior, meeting_weights):
    result = defaultdict(float)
    for previous_move, prior_probability in prior.items():
        for move, probability in meeting_weights.items():
            result[previous_move + move] += prior_probability * probability
    np.testing.assert_allclose(sum(result.values()), 1)
    return dict(result)

levels = [5.125, 5.250, 5.375]
tree = {0: 1.0}
for change in np.diff(levels) * 100:
    tree = extend_tree(tree, move_weights(change))
aggregate = move_weights(100 * (levels[-1] - levels[0]))
test = pd.DataFrame({'Conditional tree': tree, 'Aggregated': aggregate}).fillna(0).sort_index()
np.testing.assert_allclose(test.to_numpy(), [[.25, 0], [.50, 1], [.25, 0]])
display(test.rename_axis('Cumulative move by July (bp)'))

,Conditional tree,Aggregated
Cumulative move by July (bp),,
0,0.25,0.0
25,0.50,1.0
50,0.25,0.0


## 4. December 2026 and January 2027
The export ends on 11 November 2026. December and January both contain scheduled decisions, while February has none. Following CME’s anchor order, I work backwards from February through January and December. This also gives an independent check on how close the implied December starting rate is to the observed 3.88% EFFR in the team’s WIRP file.

The traditional tree uses each meeting’s model-implied change. For the Aggregated view I measure the expected terminal EFFR against 3.88%, assuming the current EFFR-to-target spread persists. The current target range in Ouiam’s notebook is 3.75–4.00%.

In [16]:
asof = pd.Timestamp('2026-11-11')
assert history.date.max() >= asof
def settlement(day, contract):
    values = history.loc[history.date.eq(day) & history.contract.eq(contract), 'px_settle']
    return float(pd.to_numeric(values, errors='raise').item())

current_rows = []
anchor = 100 - settlement(asof, 'FFG27')
for meeting, contract, anchor_id in [('2027-01-27', 'FFF27', 'FFG27'), ('2026-12-09', 'FFZ26', 'FFF27')]:
    price = settlement(asof, contract)
    average, start, end = meeting_rates(price, meeting, anchor, 'end')
    current_rows.append(dict(observation_date=asof, fomc_meeting_date=meeting, contract_id=contract,
                             instrument='30-Day Fed Funds futures', source='Bloomberg', price_type='PX_SETTLE',
                             futures_price=price, anchor_contract=anchor_id, anchor_price=settlement(asof, anchor_id),
                             anchor_side='end', anchor_effr_pct=anchor, implied_rate_pct=average,
                             start_effr_pct=start, post_effr_pct=end, expected_change_bp=100*(end-start),
                             cumulative_change_bp=100*(end-3.88)))
    anchor = start  # January's inferred starting level is December's ending level.
current = pd.DataFrame(current_rows[::-1])
display(current[['fomc_meeting_date','futures_price','start_effr_pct','post_effr_pct','expected_change_bp']].round(4))

,fomc_meeting_date,futures_price,start_effr_pct,post_effr_pct,expected_change_bp
0,2026-12-09,95.915,3.8913,4.1643,27.3004
1,2027-01-27,95.815,4.1643,4.3250,16.0741


### Meeting moves and cumulative outcomes
Each branch applies the next meeting’s move weights. Paths that end at the same cumulative move add together. The Aggregated view instead brackets the cumulative expected change from the observed EFFR directly.

In [19]:
from collections import defaultdict
previous = {0: 1.0}
current_scenarios = []
for row in current.itertuples():
    step = move_weights(row.expected_change_bp)
    tree = extend_tree(previous, step)
    previous = tree
    for view, distribution in [('meeting_move', step), ('conditional_tree', tree),
                                ('aggregated', move_weights(row.cumulative_change_bp))]:
        np.testing.assert_allclose(sum(distribution.values()), 1)
        for move, weight in distribution.items():
            current_scenarios.append([asof, row.fomc_meeting_date, view, move, weight])
current_probs = pd.DataFrame(current_scenarios, columns=['observation_date','fomc_meeting_date','view','scenario_bp','probability'])
display(current_probs.pivot(index=['fomc_meeting_date','scenario_bp'], columns='view', values='probability').round(4))

view                           aggregated  conditional_tree  meeting_move
fomc_meeting_date scenario_bp                                            
2026-12-09        25                0.863            0.9080         0.908
                  50                0.137            0.0920         0.092
2027-01-27        0                   NaN               NaN         0.357
                  25                0.220            0.3242         0.643
                  50                0.780            0.6167           NaN
                  75                  NaN            0.0592           NaN

## 5. How close is the December probability to FedWatch?
Reuters reported an 87% FedWatch probability of exactly +25 bp on 11 November. Its report gives no matched settlement timestamp or calculation view. The table tests both views. CME subsequently quoted 91.7% for +25 bp on 13 November. On 18 November Reuters described +25 bp as fully priced with a small chance of +50 bp. These later observations establish what was current when this notebook was prepared. Measuring a replication error on those dates would require new settlements.

In [22]:
dec = current_probs.loc[current_probs.fomc_meeting_date.eq('2026-12-09') & current_probs.scenario_bp.eq(25)].copy()
dec['model_pct'] = 100 * dec.probability
dec['reported_FedWatch_pct'] = 87
dec['gap_pp'] = dec.model_pct - dec.reported_FedWatch_pct
display(dec[['view','model_pct','reported_FedWatch_pct','gap_pp']].round(2))

,view,model_pct,reported_FedWatch_pct,gap_pp
0,meeting_move,90.8,87,3.8
2,conditional_tree,90.8,87,3.8
4,aggregated,86.3,87,-0.7


The December tree gives 90.80% for exactly +25 bp. The Aggregated view gives 86.30%. The first measures the move from the December starting EFFR inferred from futures. The second uses the observed 3.88% EFFR. Against the reported 87%, the gaps are +3.80 and −0.70 percentage points. The smaller gap does not establish which view Reuters observed.

### Assumptions and limits
- Rates move in multiples of 25 bp and the EFFR moves one-for-one with the target. Each mean is allocated to its two adjacent grid points.
- Within each meeting month, one rate applies through the decision date and another afterwards. Two emergency decisions in one month introduce an additional unknown rate. This limits the use of the same formula in crisis months.
- The tree applies the next meeting’s weights at each prior node. Futures alone do not identify how the second decision depends on the first.
- The workbook provides daily settlements. An exact screen comparison requires the observation time and FedWatch view.

### Outputs for the team
The first table gives implied post-meeting EFFR for each decision. The second gives scenario probabilities by view. Both retain observation and meeting dates so Ouiam and Justine can reuse them.

## 6. The full 2027 horizon as priced on 11 November 2026
The dated price snapshot supports December 2026 and all eight meetings on the 2027 schedule known that day. The Fed revised that schedule on **12 November**. Applying the revision to the previous day's settlements would introduce information traders did not have.

For March, June, September and December 2027, the preceding full non-meeting month anchors the starting rate. For April, July and October, the following non-meeting month anchors the ending rate. Independent anchor blocks can imply a small gap between consecutive endpoints. The conditional tree compounds meeting moves. The Aggregated view brackets each terminal EFFR against 3.88%. At long horizons, the distribution depends strongly on these assumptions.

In [26]:
def contract_id(month):
    return f'FF{"FGHJKMNQUVXZ"[month.month-1]}{month.year % 100:02d}'

forward = current[['observation_date','fomc_meeting_date','contract_id','futures_price','anchor_contract',
                   'anchor_price','start_effr_pct','post_effr_pct','expected_change_bp']].to_dict('records')
calendar_2027 = [('2027-03-17','start'), ('2027-04-28','end'), ('2027-06-09','start'),
                 ('2027-07-28','end'), ('2027-09-15','start'), ('2027-10-27','end'), ('2027-12-08','start')]
for meeting, side in calendar_2027:
    month = pd.Period(meeting, 'M')
    contract = contract_id(month)
    anchor_id = contract_id(month + (1 if side == 'end' else -1))
    quote, anchor_price = settlement(asof, contract), settlement(asof, anchor_id)
    _, start, end = meeting_rates(quote, meeting, 100-anchor_price, side)
    forward.append(dict(observation_date=asof, fomc_meeting_date=meeting, contract_id=contract, futures_price=quote,
                        anchor_contract=anchor_id, anchor_price=anchor_price, start_effr_pct=start,
                        post_effr_pct=end, expected_change_bp=100*(end-start)))
forward = pd.DataFrame(forward)
tree = {0:1.0}
rows = []
for row in forward.itertuples():
    steps = move_weights(row.expected_change_bp)
    tree = extend_tree(tree, steps)
    for view, weights in [('meeting_move',steps), ('conditional_tree',tree),
                          ('aggregated',move_weights(100*(row.post_effr_pct-3.88)))]:
        for move, prob in weights.items():
            rows.append([asof,row.fomc_meeting_date,view,move,prob])
forward_probs = pd.DataFrame(rows, columns=current_probs.columns)
np.testing.assert_allclose(forward_probs.groupby(['fomc_meeting_date','view']).probability.sum(),1)
display(forward[['fomc_meeting_date','contract_id','post_effr_pct','expected_change_bp']].round(3))
display(forward_probs.query("view == 'aggregated'").pivot(index='fomc_meeting_date',columns='scenario_bp',
                                                         values='probability').fillna(0).round(3))

,fomc_meeting_date,contract_id,post_effr_pct,expected_change_bp
0,2026-12-09,FFZ26,4.164,27.300
1,2027-01-27,FFF27,4.325,16.074
2,2027-03-17,FFH27,4.502,17.714
3,2027-04-28,FFJ27,4.565,6.964
4,2027-06-09,FFM27,4.651,8.571
5,2027-07-28,FFN27,4.670,2.214
6,2027-09-15,FFU27,4.690,2.000
7,2027-10-27,FFV27,4.680,-0.574
8,2027-12-08,FFZ27,4.640,-4.043


scenario_bp,25,50,75,100
fomc_meeting_date,,,,
2026-12-09,0.863,0.137,0.000,0.000
2027-01-27,0.220,0.780,0.000,0.000
2027-03-17,0.000,0.511,0.489,0.000
2027-04-28,0.000,0.260,0.740,0.000
2027-06-09,0.000,0.000,0.917,0.083
2027-07-28,0.000,0.000,0.840,0.160
2027-09-15,0.000,0.000,0.760,0.240
2027-10-27,0.000,0.000,0.800,0.200
2027-12-08,0.000,0.000,0.962,0.038


## 7. Historical replication across the supplied file
Each eligible meeting gets a **daily distribution** for the final 30 calendar days before the decision. I use a single, reproducible setup across years: the meeting contract and the following non-meeting month must both have a settlement on the observation date. This backward calculation avoids inserting a realized future EFFR. A month with another recorded meeting, conference call or exceptional event is excluded conservatively, even if the call left the rate unchanged. Observations on or before the previous recorded event are also excluded.

The calendar below transcribes Federal Reserve annual archive headings (1990–2020) and meeting calendars (2021–2026). It serves as a retrospective eligibility screen, not a real-time archive of every historical calendar revision. Early US policy changes could occur between meetings. Pre-1994 results are mechanical sensitivities under the modern convention. A scenario implying negative EFFR is flagged and withheld; near the effective lower bound even a positive-rate scenario may be economically implausible.

In [29]:
# An exclamation mark denotes a conference call, unscheduled event, notation vote or cancellation.
calendar_text = """1990 02-07 03-27 04-11! 05-15 07-03 08-21 09-07! 09-17! 10-02 11-13 12-07! 12-18
1991 01-09! 02-01! 02-06 03-26 04-12! 04-30! 05-01! 05-14 06-10! 06-24! 07-03 08-05! 08-20 09-13! 10-01 10-30! 11-05 12-02! 12-17
1992 01-09! 02-05 03-11! 03-31 05-19 07-01 07-02! 08-18 10-06 11-17 12-14! 12-22
1993 01-06! 02-03 02-18! 03-01! 03-23 05-18 07-07 08-17 09-21 10-05! 10-15! 10-22! 11-09! 11-10! 11-16 12-21
1994 02-04 02-28! 03-22 03-24! 04-18! 05-17 07-06 07-20! 08-16 09-27 11-15 12-20 12-30!
1995 01-13! 02-01 03-10! 03-28 04-28! 05-23 07-06 08-22 09-26 11-15 12-19
1996 01-31 03-26 05-21 07-03 08-20 09-24 11-13 12-17
1997 02-05 03-25 05-20 07-02 08-19 09-30 11-12 12-16
1998 02-04 03-31 05-19 07-01 08-18 09-21! 09-29 10-15! 11-17 12-22
1999 02-03 03-30 05-18 06-30 08-24 10-05 11-16 12-21
2000 02-02 03-21 05-16 06-28 08-22 10-03 11-15 12-19
2001 01-03! 01-31 03-20 04-11! 04-18! 05-15 06-27 08-21 09-13! 09-17! 10-02 11-06 12-11
2002 01-30 03-19 05-07 06-26 08-13 09-24 11-06 12-10
2003 01-29 03-18 03-25! 04-01! 04-08! 04-16! 05-06 06-25 08-12 09-15 09-16 10-28 12-09
2004 01-28 03-16 05-04 06-30 08-10 09-21 11-10 12-14
2005 02-02 03-22 05-03 06-30 08-09 09-20 11-01 12-13
2006 01-31 03-28 05-10 06-29 08-08 09-20 10-25 12-12
2007 01-31 03-21 05-09 06-28 08-07 08-10! 08-16! 09-18 10-31 12-06! 12-11
2008 01-09! 01-21! 01-30 03-10! 03-18 04-30 06-25 07-24! 08-05 09-16 09-29! 10-07! 10-29 12-16
2009 01-16! 01-28 02-07! 03-18 04-29 06-03! 06-24 08-12 09-23 11-04 12-16
2010 01-27 03-16 04-28 05-09! 06-23 08-10 09-21 10-15! 11-03 12-14
2011 01-26 03-15 04-27 06-22 08-01! 08-09 09-21 11-02 11-28! 12-13
2012 01-25 03-13 04-25 06-20 08-01 09-13 10-24 12-12
2013 01-30 03-20 05-01 06-19 07-31 09-18 10-16! 10-30 12-18
2014 01-29 03-04! 03-19 04-30 06-18 07-30 09-17 10-29 12-17
2015 01-28 03-18 04-29 06-17 07-29 09-17 10-28 12-16
2016 01-27 03-16 04-27 06-15 07-27 09-21 11-02 12-14
2017 02-01 03-15 05-03 06-14 07-26 09-20 11-01 12-13
2018 01-31 03-21 05-02 06-13 08-01 09-26 11-08 12-19
2019 01-30 03-20 05-01 06-19 07-31 09-18 10-04! 10-30 12-11
2020 01-29 03-02! 03-15! 03-18! 03-19! 03-23! 03-31! 04-29 06-10 07-29 08-27! 09-16 11-05 12-16
2021 01-27 03-17 04-28 06-16 07-28 09-22 11-03 12-15
2022 01-26 03-16 05-04 06-15 07-27 09-21 11-02 12-14
2023 02-01 03-22 05-03 06-14 07-26 09-20 11-01 12-13
2024 01-31 03-20 05-01 06-12 07-31 09-18 11-07 12-18
2025 01-29 03-19 05-07 06-18 07-30 09-17 10-29 12-10
2026 01-28 03-18 04-29 06-17 07-29 09-16 10-28 12-09"""
events = []
for line in calendar_text.splitlines():
    year, *tokens = line.split()
    for token in tokens:
        events.append((pd.Timestamp(year+'-'+token.rstrip('!')), token.endswith('!')))
events = pd.DataFrame(events, columns=['date','special']).sort_values('date')
events['month'] = events.date.dt.to_period('M')
counts = events.month.value_counts()
prices_all = history.pivot(index='date', columns='contract', values='px_settle').apply(pd.to_numeric,errors='coerce')
print(f'{len(history):,} source rows, {prices_all.shape[1]} contracts, {prices_all.index.min():%Y-%m-%d} to {prices_all.index.max():%Y-%m-%d}.')

243,913 source rows, 468 contracts, 1990-01-02 to 2026-11-11.


In [31]:
coverage, observations = [], []
for event in events.itertuples():
    meeting, month = event.date, event.month
    if meeting > asof:
        continue
    audit = dict(fomc_meeting_date=meeting, status='eligible', accepted_dates=0, floor_rejections=0)
    coverage.append(audit)
    if event.special or counts[month] != 1 or counts.get(month+1,0):
        audit['status'] = 'special event, multiple events, or next month unavailable as anchor'
        continue
    identifiers = [contract_id(month),contract_id(month+1)]
    previous = events.loc[events.date.lt(meeting),'date'].max()
    mask = (prices_all.index >= meeting-pd.Timedelta(days=30)) & (prices_all.index < meeting)
    if pd.notna(previous):
        mask &= prices_all.index > previous
    paired = prices_all.loc[mask].reindex(columns=identifiers).dropna()
    if paired.empty:
        audit['status'] = 'no common settlements'
    for observed, (quote, anchor_price) in zip(paired.index, paired.to_numpy()):
        avg,start,end = meeting_rates(quote,meeting,100-anchor_price,'end')
        change = 100*(end-start)
        distribution = move_weights(change)
        if min(start,end,start+min(distribution)/100) < -1e-9:
            audit['floor_rejections'] += 1
            continue
        np.testing.assert_allclose(sum(k*v for k,v in distribution.items()),change,atol=1e-8)
        audit['accepted_dates'] += 1
        for move,probability in distribution.items():
            observations.append([observed,meeting,*identifiers,quote,anchor_price,avg,start,end,change,move,probability])
audit = pd.DataFrame(coverage)
hist = pd.DataFrame(observations,columns=['observation_date','fomc_meeting_date','contract_id','anchor_contract',
                    'futures_price','anchor_price','implied_rate_pct','start_effr_pct','post_effr_pct',
                    'expected_change_bp','scenario_bp','probability'])
keys = ['observation_date','fomc_meeting_date']
np.testing.assert_allclose(hist.groupby(keys).probability.sum(),1)
print(f'{hist.fomc_meeting_date.nunique()} meetings and {len(hist.drop_duplicates(keys)):,} dated distributions.')
display(audit.groupby(audit.fomc_meeting_date.dt.year).agg(events_screened=('status','size'),
          meetings_with_probabilities=('accepted_dates',lambda x: (x>0).sum()), dated_distributions=('accepted_dates','sum')))

113 meetings and 2,241 dated distributions.


,events_screened,meetings_with_probabilities,dated_distributions
fomc_meeting_date,,,
1990,12,0,0
1991,19,0,0
1992,12,2,42
1993,16,2,41
1994,13,2,39
1995,11,2,36
1996,8,5,102
1997,8,4,82
1998,10,3,62


### 7.1 Financial crisis, sovereign-debt stress and COVID-19
For each of 2008, 2011 and 2020, the next table shows every eligible meeting in that year, using its last admissible pre-decision snapshot. Both scenario weights are retained. In October 2008 and March 2020 the Fed took multiple actions. A single monthly futures quote cannot identify separate probabilities for each action. In 2011 and 2020, rates near zero make the assumed 25 bp grid particularly restrictive. Blank probabilities for an excluded meeting would represent missing identification rather than a zero chance of action.

In [34]:
stress_years = [2008,2011,2020]
stress = hist.loc[hist.fomc_meeting_date.dt.year.isin(stress_years)].copy()
last = stress.groupby('fomc_meeting_date').observation_date.transform('max')
examples = stress.loc[stress.observation_date.eq(last)]
display(examples[['observation_date','fomc_meeting_date','expected_change_bp','scenario_bp','probability']].round(4))
display(audit.loc[audit.fomc_meeting_date.dt.to_period('M').isin(pd.PeriodIndex(['2008-10','2011-08','2020-03'],freq='M')),
                  ['fomc_meeting_date','status','accepted_dates']])

,observation_date,fomc_meeting_date,expected_change_bp,scenario_bp,probability
2050,2008-04-29,2008-04-30,-21.0000,-25,0.8400
2051,2008-04-29,2008-04-30,-21.0000,0,0.1600
2251,2011-01-25,2011-01-26,0.0000,0,1.0000
2291,2011-04-26,2011-04-27,1.1111,0,0.9556
2292,2011-04-26,2011-04-27,1.1111,25,0.0444
2333,2011-06-21,2011-06-22,1.3636,0,0.9455
2334,2011-06-21,2011-06-22,1.3636,25,0.0545
2336,2011-08-23,2011-09-21,0.0000,0,1.0000
3387,2020-01-28,2020-01-29,3.2069,0,0.8717
3388,2020-01-28,2020-01-29,3.2069,25,0.1283


,fomc_meeting_date,status,accepted_dates
205,2008-10-07,"special event, multiple events, or next month ...",0
206,2008-10-29,"special event, multiple events, or next month ...",0
233,2011-08-01,"special event, multiple events, or next month ...",0
234,2011-08-09,"special event, multiple events, or next month ...",0
307,2020-03-02,"special event, multiple events, or next month ...",0
308,2020-03-15,"special event, multiple events, or next month ...",0
309,2020-03-18,"special event, multiple events, or next month ...",0
310,2020-03-19,"special event, multiple events, or next month ...",0
311,2020-03-23,"special event, multiple events, or next month ...",0
312,2020-03-31,"special event, multiple events, or next month ...",0


### 7.2 Reading the three crisis cases
On 29 April 2008 the April and May settlements implied a 21 bp cut, giving 84% to a 25 bp cut and 16% to unchanged rates. In 2011 the June 21 snapshot put 94.55% on no change at the June 22 meeting. For the September meeting the last admissible snapshot was August 23, almost a month earlier. The calendar and floor checks matter when comparing those rows.

On 28 April 2020 the model put 90.69% on no change for April 29. This is a post-emergency estimate. The March emergency decisions themselves cannot be separated with the same equation. These are reconstructed probabilities under the stated assumptions; the historical cases have no matched archived CME screen in the supplied data.

### Files for the team
The historical probability export contains only model-implied meeting moves. The coverage file explains omissions. The forward export includes individual, conditional-tree and Aggregated views, all dated 11 November 2026. A later market observation requires new prices and the calendar in force on that date.

In [38]:
output = data_dir / 'fedwatch_replication'
output.mkdir(exist_ok=True)
for name, table in [('forward_rates',forward),('forward_probabilities',forward_probs),
                    ('historical_meeting_probabilities',hist),('historical_coverage',audit)]:
    table.to_csv(output / f'{name}.csv',index=False,float_format='%.9f')
print('Exported four tables.')

Exported four tables.


### Sources
- [CME Group — FedWatch methodology and September 2022 worked example](https://www.cmegroup.com/articles/2023/understanding-the-cme-group-fedwatch-tool-methodology.html)
- [CME Group — Conditional and Aggregated views](https://www.cmegroup.com/articles/2024/the-cme-fedwatch-tool-aggregated-enhancement-explained.html)
- [Federal Reserve — 2022 FOMC calendar and meeting records](https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm)
- Bloomberg daily settlements, team export. The input filenames and fields are recorded in Section 1.
- [CME FedWatch dashboard](https://www.cmegroup.com/markets/interest-rates/cme-fedwatch-tool) and [CME market update, 13 November 2026](https://www.cmegroup.com/videos/2026/11/13/90-1b-earnings-and-fomc-minutes-headline-the-week-11-13-26.html)
- Reuters, FedWatch readings on [11 November](https://www.reuters.com/world/asia-pacific/dollar-gains-traction-fed-speakers-warn-risks-inflation-2026-11-11/) and [18 November 2026](https://www.reuters.com/world/middle-east/dollar-holds-near-19-month-high-data-supports-case-us-rate-hikes-2026-11-18/). Publication times and Bloomberg settlement times can differ.
- Federal Reserve [decision and target-rate archive](https://www.federalreserve.gov/monetarypolicy/openmarket_archive.htm) and [historical meeting records](https://www.federalreserve.gov/monetarypolicy/fomc_historical.htm).
- Team Bloomberg WIRP extract, wirp_expectations_CORRECTED.csv, 11 November 2026.
- Federal Reserve [annual FOMC archives](https://www.federalreserve.gov/monetarypolicy/fomc_historical.htm), including [2008](https://www.federalreserve.gov/monetarypolicy/fomchistorical2008.htm), [2011](https://www.federalreserve.gov/monetarypolicy/fomchistorical2011.htm) and [2020](https://www.federalreserve.gov/monetarypolicy/fomchistorical2020.htm).
- Federal Reserve [2027 schedule as known on the pricing date](https://www.federalreserve.gov/newsevents/pressreleases/monetary20250905a.htm) and [subsequent revision](https://www.federalreserve.gov/newsevents/pressreleases/monetary20261112a.htm).